In [ ]:
# @title Imports
# hide noisy output from pip install
%%capture

import os
import random
import re
import time

from google.colab import drive
from google.colab import drive

import pandas as pd

import nltk
from nltk.tokenize import sent_tokenize

!pip install langchain-community langchain openai --upgrade langchain

from langchain.chat_models import ChatOpenAI
from langchain.llms import OpenAI
from langchain.chains import LLMChain
from langchain import PromptTemplate

## Setup

Due to the resource intensive model training, this notebook is intended to be run in Google Colab using an apropriate runtime.

To import the files that are required for the code below (e.g. training data files), copy them into your Google Drive and connect it to this notebook.

In [ ]:
# Input data files are stored on Google Drive to make them usable within Colab
# Mount Google Drive to access data
drive.mount("/content/drive")

DRIVE_BASE_PATH = "Please enter path to data folder" #@param

In [ ]:
# Download the punkt tokenizer if not already downloaded
nltk.download('punkt_tab')

# Set the OpenAI Key
OPENAI_AI_KEY = "Please enter an OpenAI API key" #@param
os.environ["OPENAI_API_KEY"] = OPENAI_AI_KEY

# Define train-test split ratio
TRAIN_RATIO = 0.8 # @param
# Ensures reproducibility of data generation
RANDOM_SEED = 42   # @param
# Used as post-fix for the output files

Mounted at /content/drive


# WikiBio Data Preparation

In [ ]:
# @title Load WikiBio Source Data
sent_file = f"{DRIVE_BASE_PATH}/data/train.sent"
nb_file = f"{DRIVE_BASE_PATH}/data/train.nb"

with open(nb_file, "r", encoding="utf-8") as f:
  sentence_counts = [int(line.strip()) for line in f]

with open(sent_file, "r", encoding="utf-8") as f:
  sentences = [line.strip() for line in f]

In [ ]:
# @title Reconstruct Articles
def reconstruct_articles(sentence_counts, sentences) -> pd.DataFrame:
  '''Reconstruct the articles from the input data. The input data comes as
  individual sentences and we reconstruct the articles here so we can choose
  the first two sentences.
  '''
  articles = []
  start_idx = 0

  for count in sentence_counts:
    # Get the sentences for this article
    article_sentences = sentences[start_idx:start_idx + count]
    start_idx += count

    # Combine sentences to form the paragraph
    paragraph = " ".join(article_sentences)
    articles.append(paragraph)

  return pd.DataFrame({"article": articles})


df = reconstruct_articles(sentence_counts, sentences)

# Add a column for paragraph length (e.g., word count)
df["paragraph_length"] = df["article"].apply(lambda x: len(x.split()))

# Filter paragraphs equal or greater than 150 words
filtered_paragraphs = df[(df['paragraph_length'] >= 150)]

# Shuffle and randomly pick 1000 paragraphs
sampled_paragraphs = filtered_paragraphs.sample(n=1000, random_state=RANDOM_SEED)
sampled_paragraphs.reset_index(drop=True, inplace=True)

In [ ]:
# @title Data Cleanup

def repair_parenthesis_df(df: pd.DataFrame, column: str) -> pd.DataFrame:
  '''Some paranthesis in the input data need to be fixed from '-lrb-' to '('
  and '-rrb-' to ')'.'''
  df[column] = df[column].str.replace("-lrb-", "(", regex=False)
  df[column] = df[column].str.replace("-rrb-", ")", regex=False)
  return df


def lowercase_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Converts all text in the specified column to lowercase."""
    df[column] = df[column].str.lower()
    return df

def normalize_quotes(paragraph):
  '''Replace `` and '' with standard quotation marks in the paragraphs'''
  paragraph = re.sub(r'``', '"', paragraph)  # Replace opening backticks with "
  paragraph = re.sub(r"''", '"', paragraph)  # Replace closing backticks with "
  return paragraph


## bring the brackets -lrb here----
def clean_paragraph(paragraph):
  # Normalize quotes
  paragraph = normalize_quotes(paragraph)
  # Remove space before periods
  paragraph = re.sub(r' \.', '.', paragraph)
  # Remove space before commas
  paragraph = re.sub(r' ,', ',', paragraph)
  # Remove space before closing parentheses
  paragraph = re.sub(r' \)', ')', paragraph)
  # Remove space after opening parentheses
  paragraph = re.sub(r'\( ', '(', paragraph)
  # Remove space inside quotation marks only
  paragraph = re.sub(r'"(\s+)(.*?)"', r'"\2"', paragraph)  # Remove space after opening quote
  paragraph = re.sub(r'"(.*?)\s+"', r'"\1"', paragraph)  # Remove space before closing quote
  return paragraph


sampled_paragraphs=repair_parenthesis_df(sampled_paragraphs,"article")


sampled_paragraphs_cleaned = sampled_paragraphs[['article']].copy()
sampled_paragraphs_cleaned['paragraph'] = sampled_paragraphs['article'].apply(clean_paragraph)

final_paragraph = sampled_paragraphs_cleaned[['paragraph']]
pd.set_option("display.max_colwidth", None)  # Display full text in columns


# Use the tokenizer to split the paragraphs into sentences using linguistic
# features and pick the first two sentences of each paragraph.
final_paragraph["paragraph"] = final_paragraph["paragraph"].apply(
    lambda x: " ".join(sent_tokenize(x)[:2])  # Tokenize and select first 2 sentences
)
# Sometimes the splitting by sentences fails. We filter these instances here:
# Remove rows where "paragraph" starts with ", ,, or .
final_paragraph = final_paragraph[~final_paragraph["paragraph"].str.startswith(('"', ',', '.'))]
final_paragraph = lowercase_column(final_paragraph,"paragraph")

<ipython-input-8-0e3b8cda9b7e>:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_paragraph["paragraph"] = final_paragraph["paragraph"].apply(
<ipython-input-8-0e3b8cda9b7e>:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].str.lower()


In [ ]:
final_paragraph.head(10)

,paragraph
0,ewan dowes is a rugby league footballer who has played for leeds rhinos and hull. dowes ' usual position is.
2,"humphrey adam gilbert (2 june 1886 -- 19 july 1960) was an indian-born english cricketer who played 118 first-class matches. all of these were in england, with the majority for worcestershire and oxford university."
3,"victoria ""tori"" spence (born 30 april 1984) is a new zealand stage and television actress most famous for her role as ""salene"" in the cloud 9 television drama ""the tribe"". she made her acting debut in the 1993 film ""jack be nimble"" and later appeared in the teen comedy series ""atlantis high"", in which she played the dual role of ""antonia"" and ""anthony"", and william shatner 's ""a twist in the tale"" alongside tribe co-star ryan runciman."
4,"james ralph ""sully"" montgomery (january 12, 1901 -- september 5, 1970) was an american football player and boxer. montgomery played college football for the centre praying colonels of centre college in danville, kentucky."
5,"paul blackwell (born 1954) is a well-known australian stage actor and occasional film actor. he attended the national institute of dramatic arts from 1980 to 1982 and has appeared in many productions from some of australia 's best-known theatre companies, including company b, sydney theatre company, state theatre company of south australia, patch theatre and opera australia."
6,john marriott is a former australian rules footballer who played for norwood in the sanfl. he was chosen as a ruckman in norwood 's official ` team of the century '.
7,"troy taylor is an australian rules footballer who played for richmond in the australian football league (afl). originally from central australian football league (cafl) club south alice springs, taylor was a talented junior footballer who had piqued the interest of gold coast suns football club, who sought to name taylor in their training squad in the lead up to their debut in the 2011 afl season."
8,"lari michele white (; born may 13, 1965) is an american country music artist and actress. she first gained national attention in 1992 as a winner on ""you can be a star"", a talent competition which aired on the nashville network."
9,"michael robert prior (born november 14, 1963 in chicago heights, illinois), is a former american professional football player who was selected by the tampa bay buccaneers in the 7th round of the 1985 nfl draft. a 6 ' 0 "",."
10,"william henry thompson (16 august 1927 -- 6 august 1950) was a united states army soldier and a recipient of the united states military 's highest decoration, the medal of honor, for his actions in the korean war. born to a single mother in an impoverished neighborhood in new york city, thompson entered the army in 1945 and served tours in alaska and japan."


# Data Generation

In [ ]:
def llm_chain_from_prompt_template(prompt_template):
  # Initialize the chat model
  llm = ChatOpenAI(model="gpt-4", temperature=0.7)

  # Define the prompt template
  qa_prompt = PromptTemplate(
      input_variables=["content"],
      template=prompt_template
  )

  # Create an LLMChain
  return LLMChain(llm=llm, prompt=qa_prompt)

In [ ]:
# @title Comprehension Q&A Generation

# Create 2 pairs of comprehension question and answer per paragraph

COMPREHENSION_TEMPLATE = """
You are an intelligent assistant tasked with generating high-quality comprehension-based questions and answers from the provided text.
Your goal is to create questions that assess **understanding and reasoning** rather than just direct word-matching.
The generated questions should be:

- Simple and short.
- Easy to understand.


### Instructions:
1. **Read the input paragraph carefully.**
2. **Generate two high-quality comprehension-based question** that requires understanding or reasoning about the main idea.
3. **Do NOT generate yes/no questions**—answers must provide meaningful information.
4. **The questions should be a single complete thought**, not two questions joined with "and."
5. **Avoid exact sentence repetition** from the paragraph. Instead, rephrase or summarize key ideas.
6. **Encourage inference** when applicable (e.g. ask about implications rather than just stated facts).
7. **Vary question types:** Some questions should focus on cause-effect, purpose, reasoning, or key takeaways.


### Input Text:
{content}

### Output Format:
  Q: ...?
  A: ...

  Q: ...?
  A: ...
"""

qa_chain = llm_chain_from_prompt_template(COMPREHENSION_TEMPLATE)
results = []
count=0

# Iterate over the paragraphs, generate a Q&A pair for each and return the result
# as a dataframe
for _, row in final_paragraph.iterrows():
  count += 1
  try:
    content = row["paragraph"]

    # Generate verification data from the model
    response = qa_chain.run(content=content)

    # Split response into lines and clean it
    qas = [line.strip() for line in response.split("\n") if line.strip()]

    # Extract questions and answers
    extracted_pairs = []
    current_q = None

    for line in qas:
      line = line.strip()
      if line.startswith("Q:"):  # Match "Q:" for questions
        current_q = line.split("Q: ", 1)[-1].strip()
      elif line.startswith("A:") and current_q:  # Match "A:" and ensure a question exists
        answer = line.split("A: ", 1)[-1].strip()
        extracted_pairs.append((current_q, answer))
        current_q = None  # Reset for the next question

    # Ensure we have exactly 2 Q&A pairs (1 Yes, 1 No)
    if len(extracted_pairs) == 2:
      for question, answer in extracted_pairs:
        results.append({
            "paragraph": content,
            "question": question,
            "answer": answer  # Should be "Yes" or "No"
        })
    else:
      print(f"Insufficient pairs for row {row.name}. Extracted: {extracted_pairs}")

    if count % 20 == 0: # output an update on the progress as this is rather slow
      print(f"Processed {count}/{len(final_paragraph)} paragraphs")
    time.sleep(0.5) # limit the number of requests to not get throttled by the service

  except Exception as e:
    print(f"Error for row {row.name}: {e}")

# Convert results into a DataFrame
comprehension_data = pd.DataFrame(results)

# Materialize verification data on Google Drive to avoid expensive reruns
comprehension_data.to_csv(f"{DRIVE_BASE_PATH}/data/compr_data.csv")

Processed 10/990 paragraphs
Processed 20/990 paragraphs
Processed 30/990 paragraphs
Processed 40/990 paragraphs
Processed 50/990 paragraphs
Processed 60/990 paragraphs
Processed 70/990 paragraphs
Processed 80/990 paragraphs
Processed 90/990 paragraphs
Processed 100/990 paragraphs
Processed 110/990 paragraphs
Processed 120/990 paragraphs
Processed 130/990 paragraphs
Processed 140/990 paragraphs
Processed 150/990 paragraphs
Processed 160/990 paragraphs
Processed 170/990 paragraphs
Processed 180/990 paragraphs
Processed 190/990 paragraphs
Processed 200/990 paragraphs
Processed 210/990 paragraphs
Processed 220/990 paragraphs
Processed 230/990 paragraphs
Processed 240/990 paragraphs
Processed 250/990 paragraphs
Processed 260/990 paragraphs
Processed 270/990 paragraphs
Processed 280/990 paragraphs
Processed 290/990 paragraphs
Processed 300/990 paragraphs
Processed 310/990 paragraphs
Processed 320/990 paragraphs
Processed 330/990 paragraphs
Processed 340/990 paragraphs
Processed 350/990 parag

In [ ]:
# @title Answer Verification Generation

VERIFICATION_TEMPLATE = """You are an intelligent assistant tasked with generating statement-based questions from the provided text. Each question should:
- Be a **statement** based on the provided paragraph.
- End with the phrase **"Is that correct?"**.
- Have an answer that is either **"Correct"** or **"Incorrect"**.
- 50% of the statements should be **factually correct**, and 50% should be **factually incorrect**.

### Instructions:
1. **Read the input paragraph carefully.**
2. **Generate one correct statement based on the paragraph, followed by "Is that correct?"**
   - The answer should be "Correct".
3. **Generate one incorrect statement that contradicts the paragraph, followed by "Is that correct?"**
   - The answer should be "Incorrect".
4. **Ensure the statements are evenly balanced**: 50% correct, 50% incorrect.

### Input Text:
{content}

### Output Format:
  Q: [Statement based on the paragraph], is that correct?
  A: Correct

  Q: [False statement contradicting the paragraph], is that correct?
  A: Incorrect
"""

qa_chain = llm_chain_from_prompt_template(VERIFICATION_TEMPLATE)
results = []
count = 0

# Iterate over the paragraphs, generate 2 answer verification pairs for each and return the result
# as a dataframe
for _, row in final_paragraph.iterrows():
  count += 1
  try:
    content = row["paragraph"]

    # Generate verification data from the model
    response = qa_chain.run(content=content)

    # Split response into lines and clean it
    qas = [line.strip() for line in response.split("\n") if line.strip()]

    # Extract questions and answers
    extracted_pairs = []
    current_q = None

    for line in qas:
      line = line.strip()
      if line.startswith("Q:"):  # Match "Q:" for questions
        current_q = line.split("Q: ", 1)[-1].strip()
      elif line.startswith("A:") and current_q:  # Match "A:" and ensure a question exists
        answer = line.split("A: ", 1)[-1].strip()
        extracted_pairs.append((current_q, answer))
        current_q = None  # Reset for the next question

    # Ensure we have exactly 2 Q&A pairs (1 Yes, 1 No)
    if len(extracted_pairs) == 2:
      for question, answer in extracted_pairs:
        results.append({
            "paragraph": content,
            "question": question,
            "answer": answer  # Should be "Yes" or "No"
        })
    else:
      print(f"Insufficient pairs for row {row.name}. Extracted: {extracted_pairs}")

    if count % 20 == 0: # output an update on the progress as this is rather slow
      print(f"Processed {count}/{len(final_paragraph)} paragraphs")
    time.sleep(0.5) # limit the number of requests to not get throttled by the service

  except Exception as e:
    print(f"Error for row {row.name}: {e}")

# Convert results into a DataFrame
verification_data = pd.DataFrame(results)

# Materialize verification data on Google Drive to avoid expensive reruns
verification_data.to_csv(f"{DRIVE_BASE_PATH}/data/verif_data.csv")

## Generate Final Train/Test Datasets

In [ ]:
# Split Comprehension dataset
comprehension_train = comprehension_data.sample(frac=TRAIN_RATIO, random_state=RANDOM_SEED)
comprehension_test = comprehension_data.drop(comprehension_train.index)  # Remaining 20%

# Use the entire Verification dataset in training
verification_train = verification_data.copy()  # Entire data is included in training

# Concatenate training datasets: 80% Comprehension + Full Verification
train_data = pd.concat([comprehension_train, verification_train], ignore_index=True)

# Test datasets: Remaining 20% Comprehension
test_data = comprehension_test

# Shuffle the train and test sets to prevent ordering bias
train_data = train_data.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
test_data = test_data.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Only retain relevant columns
train_data=train_data[["paragraph", "question", "answer"]]
test_data=test_data[["paragraph", "question", "answer"]]

# Convert all columns to lower case
columns_to_lowercase = ["paragraph", "question", "answer"]
train_data[columns_to_lowercase] = train_data[columns_to_lowercase].apply(lambda x: x.str.lower())
test_data[columns_to_lowercase] = test_data[columns_to_lowercase].apply(lambda x: x.str.lower())

# Store final data on Google Drive
train_data.to_csv(f"{DRIVE_BASE_PATH}/data/train_dataset.csv")
test_data.to_csv(f"{DRIVE_BASE_PATH}/data/test_dataset.csv")

In [ ]:
train_data.shape

(3560, 3)

In [ ]:
test_data.shape

(396, 3)